# Preprocessing 
**Loading Raw Data**

In [1]:
# Specifying the exact file path
file_path = "/home/imran/backup/BBCScraper/data/all.txt"

# opening and reading the file safely
with open(file_path, mode="r", encoding="utf-8") as file_handle:
    raw_text = file_handle.read()
# Checking the results
print(f"File successfully loaded! Total characters: {len(raw_text)}")


File successfully loaded! Total characters: 3541516


# Regex Cleaning
**Removal of non-Urdu characters and noise**

In [9]:
import re
def clean_urdu_text(text):
    #1. Define the range of valid Urdu Unicode characters
    urdu_range = r'[\u0600-\u06FF\u0750-\u077F\uFB50-\uFDFF\uFE70-\uFEFF]'
    # 2. Keep only valid Urdu characters and spaces.
    cleaned = re.sub(f"[^{urdu_range[1:-1]}\s]", "", text)
    # 3. Collapse multiple spaces into a single space
    cleaned = re.sub(r'\s+', ' ', cleaned)
    # 4. Strip leading and trailing whitespace from the entire document
    return cleaned.strip()

# test_input = "which essentially means that according to BBC Pakistan is a beautifull country!بی بی سی اردو: پاکستان ایک خوبصورت ملک ہے! (2026)"

# print(clean_urdu_text(test_input))    

# Normalization
Ligature & character encoding corrections.

In [10]:
def normalize_urdu_characters(text):
    # A simple dictionary replacing Arabic variants with standard Urdu Unicode equivalents
    corrections = {
        "\u0647": "\u06c1",  # Arabic Heh to Urdu Chohti Heh
        "\u064a": "\u06cc",  # Arabic Yeh to Urdu Chohti Yeh
    }
    for bad_char, good_char in corrections.items():
        text = text.replace(bad_char, good_char)
    return text

In [11]:
from IPython.display import display, HTML

def print_nastaliq(text):
    # Defining inline CSS to enforce RTL alignment, font size, and the font family
    html_output = f"""
    <p style="
        font-family: 'Jameel Noori Nastaliq', 'Awami Nastaliq', 'Urdu Typesetting', serif; 
        font-size: 28px; 
        direction: rtl; 
        text-align: right; 
        line-height: 1.8;
    ">
        {text}
    </p>
    """
    display(HTML(html_output))

# Let's test it with your cluster baseline variables!
test_bigram = "وزیر اعظم"
#print_nastaliq(f"سب سے مضبوط تعلق: {test_bigram}")

In [12]:
clean_urdu_text = clean_urdu_text(raw_text)
normalized_urdu_text = normalize_urdu_characters(clean_urdu_text)
print_nastaliq(normalized_urdu_text[:500])

# Tokenization
**Discrete unigram & bigram list creation**

In [14]:
#1. Tokenize the cleaned string into a list of words
urdu_words = normalized_urdu_text.split()
# 2. Verify the output using list slicing
print(f"Total tokens (words) in dataset: {len(urdu_words)}")
print("\nFirst 10 tokenized words:")
print_nastaliq(urdu_words[:10])


Total tokens (words) in dataset: 762377

First 10 tokenized words:


In [15]:
from collections import Counter

# Count how many times each unique word appears
unigram_counts = Counter(urdu_words)

# Let's see your vocabulary size and top words
print(f"Total Unique Words (Vocabulary Size): {len(unigram_counts)}")
print("\nTop 5 most common Urdu words in your dataset:")
print_nastaliq(unigram_counts.most_common(5))

Total Unique Words (Vocabulary Size): 26190

Top 5 most common Urdu words in your dataset:


In [16]:
# Create the bigram pairs
# zip(['A', 'B', 'C'], ['B', 'C']) -> [('A', 'B'), ('B', 'C')]
bigram_pairs = list(zip(urdu_words, urdu_words[1:]))

# Count how many times each unique pair appears
bigram_counts = Counter(bigram_pairs)

print(f"Total Unique Bigrams: {len(bigram_counts)}")
print("\nTop 5 most common word pairs:")
print_nastaliq(bigram_counts.most_common(5))

Total Unique Bigrams: 240361

Top 5 most common word pairs:


In [17]:
# Initialize our empty probability matrix
bigram_matrix = {}

# Loop through our bigram counts to calculate probabilities
for (word_a, word_b), count in bigram_counts.items():
    # Get total occurrences of Word A from our unigram counts
    total_word_a = unigram_counts[word_a]
    
    # Calculate the probability
    probability = count / total_word_a
    
    # Safely insert into our nested dictionary structure
    if word_a not in bigram_matrix:
        bigram_matrix[word_a] = {}
        
    bigram_matrix[word_a][word_b] = probability

print("\nMatrix successfully built!")


Matrix successfully built!


# Baseline 1 : Maximum Likelihood Estimates
**Constructed an MLE matrix to predict the next word based on current context**


In [21]:
test_word = "وزیر"  # Replace with a word you know exists in your text

if test_word in bigram_matrix:
    print_nastaliq(f"\nWords most likely to follow '{test_word}':")
    # Sort the following words by their probability in descending order
    sorted_followers = sorted(bigram_matrix[test_word].items(), key=lambda x: x[1], reverse=True)
    print_nastaliq(sorted_followers[:3]) # Print top 3 target words
else:
    print_nastaliq(f"'{test_word}' not found in the matrix dataset.")

# Baseline 2: Pointwise Mutual Information
- **MLE over-emphasizes highly frequent words (stop words)**  
- **PMI solves this, it is an information theoretic metric**
  

In [19]:
import math

# 1. Setup total token counts for probability calculations
total_tokens = len(urdu_words)
total_bigrams = len(bigram_pairs)

pmi_scores = {}

# 2. Loop through our existing bigram counts
for (word_a, word_b), bigram_count in bigram_counts.items():
    
    # Calculate individual unigram probabilities
    p_a = unigram_counts[word_a] / total_tokens
    p_b = unigram_counts[word_b] / total_tokens
    
    # Calculate observed bigram probability
    p_ab = bigram_count / total_bigrams
    
    # 3. Apply the PMI Formula
    # We guard against mathematical anomalies by ensuring the denominator isn't 0
    if p_a > 0 and p_b > 0:
        pmi = math.log2(p_ab / (p_a * p_b))
        pmi_scores[(word_a, word_b)] = pmi

print("PMI calculations complete!")

PMI calculations complete!


In [20]:
# Filter pairs that appear at least 5 times to prevent rare-word bias
significant_pmi = {pair: score for pair, score in pmi_scores.items() if bigram_counts[pair] >= 5}

# Sort by highest PMI score
sorted_pmi = sorted(significant_pmi.items(), key=lambda x: x[1], reverse=True)

print("\nTop 10 Strongest Word Associations (Highest PMI):")
for pair, score in sorted_pmi[:10]:
    print_nastaliq(f"Pair: {pair[0]} + {pair[1]} | PMI Score: {score:.2f}")


Top 10 Strongest Word Associations (Highest PMI):


# Baseline 3: Information Theoretic Clustering